# Step 09 — Build the Ground Truth Table

**Input:** `data/validation/sample_version_1_balanced.xlsx` (`config.VALIDATION_SOURCE_FILE`)
**Output:** `data/validation/09_ground_truth.parquet` (`config.VALIDATION_GROUND_TRUTH`)

## What this notebook does

It reads a spreadsheet that a person filled in by hand, and turns it into a tidy table
of **correct answers** — one row per building. Notebook 10 then uses that table to
measure how well the rule-based classifier performs.

## The spreadsheet

1,391 buildings. Each row arrived with a **label already filled in**, and a person went
through every row and marked their judgement using the **cell background colour** of the
next column along:

| Colour | What the person did | The correct answer is |
|---|---|---|
| 🟩 green | agreed with the label that was there | that label |
| 🟥 red | disagreed, and typed the right value into the cell | the value they typed |
| 🟨 yellow | was unsure | unknown — the row is set aside |
| no colour | did not get to this row | unknown — the row is set aside |

Green and red rows both give us a correct answer. The only difference is whether the
person had to type it out. Rows they were unsure about are set aside, and every set-aside
row is counted and explained so the totals always add up.

### Why the input is `_balanced` and not `_updated`

A building is only useful for comparing the two dimensions against each other if the
reviewer settled **both** of them. In the reviewer's working copy four rows were
green on activities but still yellow on Bosserhof, so the reviewed totals came out
uneven (899 activities vs 895 Bosserhof).

`sample_version_1_balanced.xlsx` is that file with those four rows (Excel rows 405, 431,
463 and 557) set aside on **both** dimensions — the activities cell moved from green to
yellow. The opposite fix, turning the Bosserhof cell green, was rejected on purpose: it
would assert an agreement the reviewer never gave. Only fill colours changed; no cell
value was altered.

The result is the same reviewed / set-aside split on both dimensions:

| | activities | Bosserhof |
|---|---|---|
| reviewed (green + red) | 895 | 895 |
| uncertain | 337 | 337 |
| unvalidated | 159 | 159 |

Note that **equal verdict counts do not mean equal scoreable counts** — Steps 3 and 4
drop further rows for reasons unrelated to colour. See the end of Step 4.

## Two kinds of label per building

| | Shape | Example |
|---|---|---|
| **Activities** | a **set** — a building can host several | `{Workers, Retail_Daily}` |
| **Bosserhof class** | a **single** value | `retail small scale` |

An empty set means "nothing happens here"; for Bosserhof, `''` means "this building gets
no class". Both are real answers, not missing data.

## A note on reading the columns

The sheet has eight columns and **two of them share the same header text**
(`bosserhof_class_clean` — the pre-filled value in C, the person's verdict in D). So the
columns are read by **position** rather than by name. Looking them up by name makes both
collapse onto the same column, which loses one of them silently.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_SOURCE_FILE, VALIDATION_GROUND_TRUTH, VALIDATION_COLOURS,
    VALIDATION_NO_ACTIVITY_TERMS, VALIDATION_ACTIVITY_TYPO_FIXES,
    ZONE_ACTIVITY_COLUMNS, BOSSERHOF_WEIGHTS, BOSSERHOF_NORMALIZATION_MAP,
)
from rule_utils import reachable_bosserhof_classes

import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

pd.set_option('display.width', 170)
pd.set_option('display.max_colwidth', 60)

if not VALIDATION_SOURCE_FILE.exists():
    raise FileNotFoundError(f'{VALIDATION_SOURCE_FILE} not found')
print(f'Source: {VALIDATION_SOURCE_FILE.name}')

Source: sample_version_1_balanced.xlsx


---
## Step 1 — Check the spreadsheet layout

Print the header of each of the eight columns and compare it with what the rest of the
notebook expects. If someone opens this on a differently arranged sheet, it stops right
here with a clear message about which column moved.

In [2]:
wb = load_workbook(VALIDATION_SOURCE_FILE)
ws = wb.active
print(f'sheet {ws.title!r}: {ws.max_row - 1:,} data rows x {ws.max_column} columns\n')

# Fixed positions (1-indexed), NOT header lookup - columns C and D share a header.
COL_ACT_PREFILL  = 1   # A  pre-filled activity set
COL_ACT_VERDICT  = 2   # B  verdict colour + the human replacement, if any
COL_BOSS_PREFILL = 3   # C  pre-filled Bosserhof class
COL_BOSS_VERDICT = 4   # D  verdict colour + the human replacement, if any
COL_OSM_NAMES    = 5   # E
COL_GML_ID       = 6   # F
COL_TARGET_TAZ   = 7   # G
COL_VOLUME_M3    = 8   # H

EXPECTED_HEADERS = {
    COL_ACT_PREFILL:  'mid_label',
    COL_ACT_VERDICT:  'mid_label_new',
    COL_BOSS_PREFILL: 'bosserhof_class_clean',
    COL_BOSS_VERDICT: 'bosserhof_class_clean',   # yes, the same text as C
    COL_OSM_NAMES:    'osm_names',
    COL_GML_ID:       'gml_id',
    COL_TARGET_TAZ:   'target_taz',
    COL_VOLUME_M3:    'volume_m3',
}

print(f"{'col':>4}  {'actual header':<24} {'expected':<24} status")
mismatches = []
for idx, expected in EXPECTED_HEADERS.items():
    actual = ws.cell(1, idx).value
    if actual != expected:
        mismatches.append((get_column_letter(idx), actual, expected))
    print(f'{get_column_letter(idx):>4}  {str(actual):<24} {expected:<24} '
          f'{"OK" if actual == expected else "MISMATCH"}')

assert not mismatches, (
    f'Column layout differs from what this notebook assumes: {mismatches}. '
    'Fix the COL_* positions above - do not run on a guess.'
)
print('\nLayout confirmed.')

sheet 'Tabelle1': 1,391 data rows x 8 columns

 col  actual header            expected                 status
   A  mid_label                mid_label                OK
   B  mid_label_new            mid_label_new            OK
   C  bosserhof_class_clean    bosserhof_class_clean    OK
   D  bosserhof_class_clean    bosserhof_class_clean    OK
   E  osm_names                osm_names                OK
   F  gml_id                   gml_id                   OK
   G  target_taz               target_taz               OK
   H  volume_m3                volume_m3                OK

Layout confirmed.


---
## Step 2 — Read the colours

The person's judgement is stored as a **cell colour**, so we read the fill of each
verdict cell and translate it into a word. `config.VALIDATION_COLOURS` holds the
three colour codes they used:

| Colour code | Meaning |
|---|---|
| `FFC6EFCE` | agreed |
| `FFFFC7CE` | disagreed |
| `FFFFEB9C` | unsure |

Anything else, including an uncoloured cell, means the row was never reviewed.

Each building gets **two separate judgements** — one for its activities, one for its
Bosserhof class — because the person could agree with one and disagree with the other.
The cross-tab at the end of the cell shows how often that happened.

In [3]:
def cell_verdict(cell):
    """Read the verdict off a cell fill colour."""
    fill = cell.fill
    rgb = fill.fgColor.rgb if (fill is not None and fill.fgColor is not None) else None
    if not isinstance(rgb, str):
        return 'unvalidated'
    return VALIDATION_COLOURS.get(rgb.upper(), 'unvalidated')


records = []
for r in range(2, ws.max_row + 1):
    gml_id = ws.cell(r, COL_GML_ID).value
    if gml_id is None:
        continue
    records.append({
        'gml_id':            str(gml_id).strip(),
        'row_in_workbook':   r,
        'act_prefill_raw':   ws.cell(r, COL_ACT_PREFILL).value,
        'act_replacement_raw': ws.cell(r, COL_ACT_VERDICT).value,
        'act_verdict':       cell_verdict(ws.cell(r, COL_ACT_VERDICT)),
        'boss_prefill_raw':  ws.cell(r, COL_BOSS_PREFILL).value,
        'boss_replacement_raw': ws.cell(r, COL_BOSS_VERDICT).value,
        'boss_verdict':      cell_verdict(ws.cell(r, COL_BOSS_VERDICT)),
        'osm_names':         ws.cell(r, COL_OSM_NAMES).value,
        'target_taz':        ws.cell(r, COL_TARGET_TAZ).value,
        'volume_m3':         ws.cell(r, COL_VOLUME_M3).value,
    })

df = pd.DataFrame(records)
print(f'{len(df):,} rows with a gml_id  |  unique: {df["gml_id"].nunique():,}')
assert df['gml_id'].is_unique, 'gml_id is not unique - downstream joins would fan out'

print('\nactivity verdicts:')
print(df['act_verdict'].value_counts().to_string())
print('\nbosserhof verdicts:')
print(df['boss_verdict'].value_counts().to_string())
print('\nThe two verdicts are independent, not one judgement applied twice:')
print(pd.crosstab(df['act_verdict'], df['boss_verdict'],
                  rownames=['activities'], colnames=['bosserhof']).to_string())

1,391 rows with a gml_id  |  unique: 1,391

activity verdicts:
act_verdict
correct        553
error          342
uncertain      337
unvalidated    159

bosserhof verdicts:
boss_verdict
correct        719
uncertain      337
error          176
unvalidated    159

The two verdicts are independent, not one judgement applied twice:
bosserhof    correct  error  uncertain  unvalidated
activities                                         
correct          503     50          0            0
error            216    126          0            0
uncertain          0      0        337            0
unvalidated        0      0          0          159


---

---
## Step 3 — Work out the correct activities

There are only **7 possible activities** (`config.ZONE_ACTIVITY_COLUMNS`):

`Workers` · `Retail_Daily` · `Retail_Non-Daily` · `Leisure` · `School` · `University` · `Kindergarten`

Because that list is short and fixed, we scan each cell for words from it rather than
trying to parse the cell as a Python list. That matters: about one in six of the typed
answers has a broken bracket or quote (`['Retail_Non-Daily]`,
`['Workers'; Retail_Non-Daily']`), and word-scanning reads those fine.

Three situations give us a correct answer:

| Situation | Correct answer |
|---|---|
| 🟩 agreed | the activities that were already in the cell |
| 🟥 disagreed, typed a replacement | the activities they typed |
| 🟥 disagreed, wrote "residential" or "living" | **the empty set** — nothing happens in this building |

That third case is worth spelling out: an empty set is a genuine answer, not missing
data. It says the classifier claimed an activity where there is none, which is exactly
what we want to be able to measure.

A red cell left blank means the person flagged a problem but never wrote the fix, so
there is no answer to compare against and the row is set aside.

### Catching typos

If the person wrote `Kindergarden` instead of `Kindergarten`, word-scanning finds
nothing and their label quietly disappears from the answer. Two such typos exist in this
spreadsheet, so `config.VALIDATION_ACTIVITY_TYPO_FIXES` lists the misspellings to accept.

The check at the end of this cell looks for any other word that resembles an activity
name and stops the notebook if it finds one, naming the building and the cell. That way
a new typo gets fixed rather than silently losing a label.

In [4]:
CANONICAL_ACTIVITIES = {}
for name in ZONE_ACTIVITY_COLUMNS:
    key = name.lower()
    CANONICAL_ACTIVITIES[key] = name
    CANONICAL_ACTIVITIES[key.replace('-', '_')] = name
    CANONICAL_ACTIVITIES[key.replace('_', '-')] = name
for typo, canonical in VALIDATION_ACTIVITY_TYPO_FIXES.items():
    assert canonical in ZONE_ACTIVITY_COLUMNS, (
        f'typo fix maps {typo!r} to {canonical!r}, not one of {ZONE_ACTIVITY_COLUMNS}')
    CANONICAL_ACTIVITIES[typo.lower()] = canonical


def parse_activities(raw):
    """Canonical activity names found in a hand-typed cell, or None if unusable."""
    if raw is None:
        return None
    text = str(raw).strip()
    if text == '' or text.lower() in ('none', 'nan', '[]'):
        return None
    found = []
    for token in re.findall(r'[A-Za-z][A-Za-z_\-]*', text):
        name = CANONICAL_ACTIVITIES.get(token.lower())
        if name is not None and name not in found:
            found.append(name)
    return sorted(found) if found else None


def says_no_activity(raw):
    """True when the human wrote free text meaning the building hosts no activity."""
    words = re.sub(r'[^a-z ]+', ' ', str(raw or '').lower()).split()
    if not words:
        return False
    return (' '.join(words) in VALIDATION_NO_ACTIVITY_TERMS
            or 'living' in words or 'residential' in words)


def resembles_activity(token):
    """True when an unrecognised token looks like a garbled activity name.

    Used only to raise the alarm, never to guess a value.
    """
    t = token.lower()
    if t in CANONICAL_ACTIVITIES:
        return False
    return any(len(t) >= 4 and (t[:4] == known[:4] or known.startswith(t[:5]))
               for known in CANONICAL_ACTIVITIES)


def resolve_activities(row):
    """Returns (truth, reason). truth is None when the row cannot be scored."""
    verdict = row['act_verdict']
    if verdict in ('uncertain', 'unvalidated'):
        return None, f'excluded: {verdict}'
    if verdict == 'correct':
        confirmed = parse_activities(row['act_prefill_raw'])
        if confirmed is None:
            return None, 'excluded: confirmed label could not be parsed'
        return confirmed, 'human confirmed the label'
    replacement = parse_activities(row['act_replacement_raw'])
    if replacement:
        return replacement, 'human typed a replacement'
    if says_no_activity(row['act_replacement_raw']):
        return [], 'human wrote: no activity here'
    return None, 'excluded: marked wrong but never corrected'


_res = df.apply(resolve_activities, axis=1, result_type='expand')
df['activities_truth'] = _res[0]
df['activities_scoreable_reason'] = _res[1]
df['activities_scoreable'] = df['activities_truth'].notna()

print(df['activities_scoreable_reason'].value_counts().to_string())
print(f'\nActivities scoreable: {int(df["activities_scoreable"].sum()):,} of {len(df):,}')

# --- GUARD: a misspelled activity name must never be dropped silently -----------
suspect = {}
for col in ('act_prefill_raw', 'act_replacement_raw'):
    for gml_id, raw in zip(df['gml_id'], df[col]):
        if raw is None:
            continue
        for token in re.findall(r'[A-Za-z][A-Za-z_\-]*', str(raw)):
            if resembles_activity(token):
                suspect.setdefault(token, []).append((gml_id, str(raw)))

if suspect:
    print('\nUNRECOGNISED TOKENS THAT LOOK LIKE ACTIVITY NAMES:')
    for token, hits in suspect.items():
        print(f'  {token!r}  x{len(hits)}')
        for gml_id, raw in hits[:3]:
            print(f'      gml_id={gml_id}  {raw!r}')
    raise AssertionError(
        f'{sorted(suspect)} resemble activity names but match nothing. Each is a label '
        'the human wrote that would be silently dropped from the ground truth. Add the '
        'spelling to VALIDATION_ACTIVITY_TYPO_FIXES in config.py, or confirm it is not '
        'an activity name.')
print('\nGUARD PASSED - no misspelled activity name is being dropped.')

activities_scoreable_reason
human confirmed the label                     553
excluded: uncertain                           337
human typed a replacement                     316
excluded: unvalidated                         159
excluded: marked wrong but never corrected     13
human wrote: no activity here                  13

Activities scoreable: 882 of 1,391

GUARD PASSED - no misspelled activity name is being dropped.


---
## Step 4 — Work out the correct Bosserhof class

One class per building, chosen from the 47 in `config.BOSSERHOF_WEIGHTS`. These answers
were typed freehand, so they arrive in four shapes:

**1. Almost-right spellings** — `restaurant gastronomy`, `small scale retail`, `nursing`,
`diy store`. A plural or a word order away from a real class name. Corrected
automatically.

**2. "No class here"** — `none`, `seems residential`, `vacancy`, `just garages`. The
person is saying this building should not get a Bosserhof class at all. Stored as `''`,
which is a real answer we can score against.

**3. Several classes at once** — `kindergartens public facilities`, or
`restaurants gastronomy customer oriented services entertainment culture`. The person
named two or more because the building genuinely has several uses. Bosserhof allows only
one class per building, so there is no single answer to score against and the row is set
aside.

We spot these by counting how many known class names appear inside the text, rather than
keeping a list of specific phrases — so it works on wordings this spreadsheet happens
not to contain.

**4. Something the vocabulary has no word for** — `garbage collection`, `fraternity`.
Kept exactly as written and reported in Step 5.

In [5]:
BOSSERHOF_KNOWN = {k.lower() for k in BOSSERHOF_WEIGHTS}

BOSSERHOF_TYPO_FIXES = {
    'restaurant gastronomy':      'restaurants gastronomy',
    'customer oriented service':  'customer oriented services',
    'business oriented service':  'business oriented services',
    'business oriented business': 'business oriented services',
    'school':                     'schools',
    'kindergarten':               'kindergartens',
    'hospital':                   'hospitals',
    'hotel':                      'hotels',
    'university':                 'universities',
    'research institute':         'research institutes',
    'nursing':                    'nursing homes',
    'nursing home':               'nursing homes',
    'craft business':             'craft businesses',
    'diy store':                  'diy stores',
    'shopping center':            'shopping centers',
    'small scale retail':         'retail small scale',
}
BOSSERHOF_NO_CLASS_TERMS = {'none', 'no', 'vacancy', 'vacant', 'just garages', 'garages'}
BOSSERHOF_SUBSTRING_FIXES = {
    r'\brestaurant gastronomy\b':     'restaurants gastronomy',
    r'\bleisture\b':                  'leisure',
    r'\bcustomer oriented service\b': 'customer oriented services',
    r'\bbusiness oriented service\b': 'business oriented services',
}


def clean_bosserhof(raw):
    """Normalise a Bosserhof string. '' means an explicit no-class; None unusable."""
    if raw is None:
        return None
    text = re.sub(r'[^a-z0-9 ]+', ' ', str(raw).lower())
    text = re.sub(r'\s+', ' ', text).strip()
    if text in ('', 'nan'):
        return None
    if (text in VALIDATION_NO_ACTIVITY_TERMS or text in BOSSERHOF_NO_CLASS_TERMS
            or 'residential' in text):
        return ''
    for pattern, replacement in BOSSERHOF_SUBSTRING_FIXES.items():
        text = re.sub(pattern, replacement, text)
    text = BOSSERHOF_TYPO_FIXES.get(text, text)
    return BOSSERHOF_NORMALIZATION_MAP.get(text, text)


def classes_mentioned(text):
    """Known classes appearing as substrings, longest first so a class containing
    another (retail small scale vs retail) is not double-counted."""
    if not text:
        return []
    found, remaining = [], text
    for known in sorted(BOSSERHOF_KNOWN, key=len, reverse=True):
        if known in remaining:
            found.append(known)
            remaining = remaining.replace(known, ' ')
    return found


def resolve_bosserhof(row):
    """Returns (truth, reason). truth is None when the row cannot be scored."""
    verdict = row['boss_verdict']
    if verdict in ('uncertain', 'unvalidated'):
        return None, f'excluded: {verdict}'

    value = (clean_bosserhof(row['boss_prefill_raw']) if verdict == 'correct'
             else clean_bosserhof(row['boss_replacement_raw']))
    if value is None:
        return None, ('excluded: confirmed class could not be parsed' if verdict == 'correct'
                      else 'excluded: marked wrong but never corrected')
    if value == '':
        return '', 'human wrote: no class'
    if value in BOSSERHOF_KNOWN:
        return value, ('human confirmed the class' if verdict == 'correct'
                       else 'human typed a replacement')

    mentioned = classes_mentioned(value)
    if len(mentioned) >= 2:
        return None, 'excluded: human named several classes, no single truth'
    if len(mentioned) == 1:
        return mentioned[0], 'single class extracted from free text'
    # Kept verbatim; Step 5 reports it. The human described a use the Bosserhof
    # vocabulary has no name for.
    return value, 'human wrote a value that is not a Bosserhof class'


_res = df.apply(resolve_bosserhof, axis=1, result_type='expand')
df['bosserhof_truth'] = _res[0]
df['bosserhof_scoreable_reason'] = _res[1]
df['bosserhof_scoreable'] = df['bosserhof_truth'].notna()

print(df['bosserhof_scoreable_reason'].value_counts().to_string())
print(f'\nBosserhof scoreable: {int(df["bosserhof_scoreable"].sum()):,} of {len(df):,}')
print(f'  of which truth is "" (no class): {int((df["bosserhof_truth"] == "").sum()):,}')

bosserhof_scoreable_reason
human confirmed the class                                 719
excluded: uncertain                                       337
excluded: unvalidated                                     159
human typed a replacement                                 131
human wrote: no class                                      14
excluded: marked wrong but never corrected                 13
excluded: human named several classes, no single truth      8
single class extracted from free text                       7
human wrote a value that is not a Bosserhof class           3

Bosserhof scoreable: 874 of 1,391
  of which truth is "" (no class): 14


---
## Step 5 — Answers the rules can never give

`rule_utils` can only ever output a class that one of its rules points to. So if the
correct answer for a building is a class no rule points to, the classifier will get that
building wrong no matter how good the rules are.

Worth separating from ordinary mistakes, because the fix is different:

| Situation | What it means | What would fix it |
|---|---|---|
| the answer is a valid Bosserhof class, but no rule points to it | the class exists in `config.BOSSERHOF_WEIGHTS` and is simply unused | **add a rule** |
| the answer is not a Bosserhof class at all | the person described a use the 47 classes have no word for | **a taxonomy decision** |

This cell lists both, building by building, so they can be acted on.

In [6]:
REACHABLE = reachable_bosserhof_classes()

unreachable_defined = sorted(BOSSERHOF_KNOWN - REACHABLE)
print(f'{len(unreachable_defined)} of {len(BOSSERHOF_KNOWN)} defined Bosserhof classes '
      'are unreachable - no rule in rule_utils emits them:')
for cls in unreachable_defined:
    print(f'   {cls}')

scoreable = df[df['bosserhof_scoreable']].copy()
scoreable['producible'] = scoreable['bosserhof_truth'].map(
    lambda v: True if v == '' else v.lower() in REACHABLE)

blocked = scoreable[~scoreable['producible']]
print(f'\n{len(blocked)} of {len(scoreable):,} scoreable buildings have a ground-truth '
      'class the rules CANNOT produce:')
if len(blocked):
    for _, row in blocked.iterrows():
        kind = ('valid class, but no rule emits it' if row['bosserhof_truth'].lower()
                in BOSSERHOF_KNOWN else 'not a Bosserhof class at all')
        print(f'   gml_id={row["gml_id"]:>8}  truth={row["bosserhof_truth"]!r:34} ({kind})')
        print(f'                     name={str(row["osm_names"])[:60]!r}')
else:
    print('   none')

df['bosserhof_truth_producible'] = df['bosserhof_truth'].map(
    lambda v: pd.NA if v is None else (True if v == '' else v.lower() in REACHABLE))

# Activities use a closed 7-name vocabulary that the classifier always covers, so
# there is nothing equivalent to report - assert that rather than assume it.
from config import MID_LABEL_TO_ACTIVITY
_covered = set(MID_LABEL_TO_ACTIVITY.values())
_missing = set(ZONE_ACTIVITY_COLUMNS) - _covered
assert not _missing, f'activities the classifier can never emit: {_missing}'
print(f'\nActivities: all {len(ZONE_ACTIVITY_COLUMNS)} zone activities are producible '
      'by the classifier - nothing blocked on that dimension.')

7 of 47 defined Bosserhof classes are unreachable - no rule in rule_utils emits them:
   craft courtyards
   crafts and trades
   customer service
   factory outlet centers
   retail wholesale
   self service department stores
   suppliers for car dealerships

4 of 874 scoreable buildings have a ground-truth class the rules CANNOT produce:
   gml_id=  550941  truth='garbage collection'               (not a Bosserhof class at all)
                     name="['Kreiswirtschaftsbetriebe Goslar']"
   gml_id=  231906  truth='factory outlet centers'           (valid class, but no rule emits it)
                     name='[\'Doki Kinderbetreuung\', \'NAME IT\', "O\'Neill", \'Seidensticke'
   gml_id=  345680  truth='better school'                    (not a Bosserhof class at all)
                     name="['Sporthalle des Gymnasiums am Silberkamp']"
   gml_id=  556702  truth='fraternity'                       (not a Bosserhof class at all)
                     name="['Corps Rhenania ZAB']"

Ac

---
## Step 6 — Save

The saved table keeps the correct answers and enough information to identify each
building:

| Column | Holds |
|---|---|
| `gml_id`, `row_in_workbook` | which building, and which spreadsheet row it came from |
| `osm_names`, `target_taz`, `volume_m3` | building details, useful when reading results |
| `activities_truth` | the correct set of activities |
| `activities_scoreable`, `..._reason` | whether it can be used, and why not if it cannot |
| `bosserhof_truth` | the correct Bosserhof class |
| `bosserhof_scoreable`, `..._reason` | same, for the single-value dimension |
| `bosserhof_truth_producible` | whether any rule can output that class |

`volume_m3` earns its place: notebook 10 uses it to confirm the building ids still point
at the same buildings before it compares anything.

The cell also runs a few checks — ids are unique, every usable row has an answer, and
the pre-filled values from the spreadsheet have been left behind — so the table is
either right or it stops here.

In [7]:
OUTPUT_COLUMNS = [
    # identity
    'gml_id', 'row_in_workbook', 'osm_names', 'target_taz', 'volume_m3',
    # activities - the human answer
    'activities_truth', 'activities_scoreable', 'activities_scoreable_reason',
    # bosserhof - the human answer
    'bosserhof_truth', 'bosserhof_scoreable', 'bosserhof_scoreable_reason',
    'bosserhof_truth_producible',
]
out = df[OUTPUT_COLUMNS].copy()

# --- correctness of the table itself ---
assert out['gml_id'].is_unique, 'gml_id not unique'
assert out.loc[out['activities_scoreable'], 'activities_truth'].notna().all(), \
    'a scoreable row is missing its activity truth'
assert out.loc[~out['activities_scoreable'], 'activities_truth'].isna().all(), \
    'an unscoreable row is carrying an activity truth'
assert out.loc[out['bosserhof_scoreable'], 'bosserhof_truth'].notna().all(), \
    'a scoreable row is missing its bosserhof truth'

# --- nothing that could be mistaken for a prediction ---
_pred_like = [c for c in out.columns
              if 'prefill' in c or 'replacement' in c or 'predicted' in c]
assert not _pred_like, f'prediction columns leaked into the ground truth: {_pred_like}'

# --- nothing that could be mistaken for a result ---
_FORBIDDEN = ('precision', 'recall', 'f1', 'accuracy', 'n_extra', 'n_missing',
              'true_positive', 'over_predicted', 'exact_match', 'correct')
_metric_like = [c for c in out.columns if any(f in c.lower() for f in _FORBIDDEN)]
assert not _metric_like, f'metric columns leaked into the ground truth: {_metric_like}'

print('Assertions passed: ground truth only, no predictions, no metrics.\n')

# Flat string form alongside the list column - easier to eyeball, and immune to
# how parquet handles mixed-type object columns.
out['activities_truth_str'] = out['activities_truth'].map(
    lambda v: '' if v is None else ';'.join(v) if isinstance(v, list) else str(v))

VALIDATION_GROUND_TRUTH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(VALIDATION_GROUND_TRUTH, index=False)

print(f'Saved {len(out):,} rows x {len(out.columns)} columns -> {VALIDATION_GROUND_TRUTH}')
print(f'  activities scoreable : {int(out["activities_scoreable"].sum()):,}')
print(f'  bosserhof  scoreable : {int(out["bosserhof_scoreable"].sum()):,}')
print()
print('columns:')
for c in out.columns:
    print(f'   {c}')

Assertions passed: ground truth only, no predictions, no metrics.

Saved 1,391 rows x 13 columns -> C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\validation\09_ground_truth.parquet
  activities scoreable : 882
  bosserhof  scoreable : 874

columns:
   gml_id
   row_in_workbook
   osm_names
   target_taz
   volume_m3
   activities_truth
   activities_scoreable
   activities_scoreable_reason
   bosserhof_truth
   bosserhof_scoreable
   bosserhof_scoreable_reason
   bosserhof_truth_producible
   activities_truth_str


In [8]:
out.head(12)

,gml_id,row_in_workbook,osm_names,target_taz,volume_m3,activities_truth,activities_scoreable,activities_scoreable_reason,bosserhof_truth,bosserhof_scoreable,bosserhof_scoreable_reason,bosserhof_truth_producible,activities_truth_str
0,236928,2,['Enoteca Vetrone'],Rühen 7_472,1554.252341,"[Leisure, Workers]",True,human confirmed the label,restaurants gastronomy,True,human confirmed the class,True,Leisure;Workers
1,535772,3,None,WOB Ehmen 2_289,453.347111,None,False,excluded: uncertain,None,False,excluded: uncertain,<NA>,
2,477334,4,None,Gifhorn 04_416,727.942206,None,False,excluded: uncertain,None,False,excluded: uncertain,<NA>,
3,388629,5,"[""Deutsche Bank;Ernsting's family""]",Goslar 18_565,17437.047953,"[Retail_Non-Daily, Workers]",True,human typed a replacement,retail small scale,True,human confirmed the class,True,Retail_Non-Daily;Workers
4,81399,6,None,Bad Harzburg Schlewecke_518,248.752010,None,False,excluded: uncertain,None,False,excluded: uncertain,<NA>,
5,81391,7,None,Bad Harzburg Schlewecke_520,726.737078,[Workers],True,human confirmed the label,industrial operations production,True,human confirmed the class,True,Workers
6,36228,8,"['Nazar Trockenfrüchte', ""Sara's Collection"", 'Science a...",BS Stadtkern 13_2,12947.777792,"[Retail_Daily, Retail_Non-Daily, University, Workers]",True,human typed a replacement,retail small scale,True,human confirmed the class,True,Retail_Daily;Retail_Non-Daily;University;Workers
7,556581,9,['Reni'],Sankt Andreasberg 04_547,1371.218910,"[Leisure, Workers]",True,human confirmed the label,hotels,True,human confirmed the class,True,Leisure;Workers
8,389444,10,None,Braunlage 02_545,105.044662,"[Retail_Daily, Retail_Non-Daily, Workers]",True,human typed a replacement,customer oriented services,True,human confirmed the class,True,Retail_Daily;Retail_Non-Daily;Workers
9,453160,11,None,Goslar 05_552,1515.565616,[Retail_Non-Daily],True,human typed a replacement,business oriented services,True,human typed a replacement,True,Retail_Non-Daily


---
## Done

`09_ground_truth.parquet` now holds one row per reviewed building with the correct
answer for each dimension.

**Next:** `10_validation_scoring.ipynb` runs the rule-based classifier over these same
buildings and compares its output against these answers, reporting precision, recall and
accuracy.